In [1]:
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

In [2]:
from graphviz import Digraph

## Kaggle Notebook ##
- Here is the [NoteBook](https://www.kaggle.com/code/dbais001/decisiontree-numpy-simple-code/edit)
- Medium post https://medium.com/nailing-the-ai-ml-interview/ml-coding-interview-decision-trees-4eb946294bb7

In [3]:
## Lets take some information of the tree ## 

class TreeNode:
    def __init__(self, feature = None, threshold = None, val = None, left = None, right = None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.val = val

    def __str__(self):
        left_str = str(self.left) if self.left else "None"
        right_str = str(self.right) if self.right else "None"
        return f"TreeNode(val={self.val}, left = {left_str}, right ={right_str})"

In [4]:
node = TreeNode()
node.val = 1
node.left = TreeNode(val = 2, left = 3, right = 4)

In [5]:
print(node)

TreeNode(val=1, left = TreeNode(val=2, left = 3, right =4), right =None)


In [56]:
class MyDecisionTreeClassifier:
    def __init__(self, max_depth = 6):
        self.max_depth = max_depth
        self.Tree = None
        
        
    def gini(self, y):
        """Calculate the Gini impurity for a list of classes."""
        
        total = len(y)
        if total == 0:
            return 0
        counts = Counter(y)
        impurity = 1 - sum((count/total) ** 2 for count in counts.values())
        return impurity
        
    def best_split(self, X, y):
        """Find the best feature and threshold to split the data."""
        
        best_gini = float('inf')
        best_feature = None
        best_threshold = None
        n_features = X.shape[1]

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature]) 
            for threshold in thresholds:
                left_indices = X[:, feature] < threshold
                right_indices = X[:, feature] >= threshold
                
                if np.any(left_indices) and np.any(right_indices):
                    left_classes = y[left_indices]
                    right_classes = y[right_indices]
                    
                    gini_left = self.gini(left_classes)
                    gini_right = self.gini(right_classes)
                    weighted_gini = (len(left_classes) * gini_left + len(right_indices) * gini_right) / len(y)

                    if weighted_gini< best_gini:
                        best_gini = weighted_gini
                        best_feature = feature
                        best_threshold = threshold
                        
        return best_feature, best_threshold

    def fit(self, X, y, depth = 0):
        """Recursively build the decision tree."""
        
        if depth == self.max_depth or len(set(y)) == 1:
            return TreeNode(val = Counter(y).most_common(1)[0][0])
            
        feature, threshold = self.best_split( X, y)

        if feature is None:
            return TreeNode(val = Counter(y).most_common(1)[0][0]) # majority element in y 
            
        left_indices = X[:, feature] < threshold
        right_indices = X[:, feature] >= threshold

        
        left_node = self.fit(X[left_indices], y[left_indices], depth + 1)
        right_node = self.fit(X[right_indices], y[right_indices], depth+1)
        
        self.Tree = TreeNode(feature=feature, threshold=threshold, left=left_node, right=right_node)
        return self.Tree


    def predict_one(self, node, X):
        """Predict the class for a single sample."""
        if node.val is not None:
            return node.val
        if X[node.feature] < node.threshold:
            return self.predict_one(node.left, X)
        else:
            return self.predict_one(node.right, X)

    def predict(self, X):
        
        """ Predict the classes for a set of samples. """
        return np.array([self.predict_one(self.Tree, x) for x in X])
        
    def plot_tree(self, tree=None, parent=None, graph=None):
        """
        [Optional]Visualize the decision tree structure using Graphviz.
        """
        if graph is None:
            graph = Digraph()
            tree = self.Tree if tree is None else tree
        
        if isinstance(tree, dict):
            for key, value in tree.items():
                node_id = f"node_{key}"
                graph.node(node_id, key)
                if parent:
                    graph.edge(parent, node_id)
                self.plot_tree(value, node_id, graph)
        else:
            leaf_id = f"leaf_{tree}"
            graph.node(leaf_id, f"Class {tree}", shape='box')
            if parent:
                graph.edge(parent, leaf_id)
        
        return graph

In [57]:
train = pd.read_csv("./playground-series-s5e7/train.csv")
test = pd.read_csv("./playground-series-s5e7/test.csv")
submission = pd.read_csv("./playground-series-s5e7/sample_submission.csv")

In [58]:
train.head()

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0,0.0,No,6.0,4.0,No,15.0,5.0,Extrovert
1,1,1.0,No,7.0,3.0,No,10.0,8.0,Extrovert
2,2,6.0,Yes,1.0,0.0,NaN,3.0,0.0,Introvert
3,3,3.0,No,7.0,3.0,No,11.0,5.0,Extrovert
4,4,1.0,No,4.0,4.0,No,13.0,NaN,Extrovert


### Encode target value

In [59]:
le = LabelEncoder()
train["Personality_encoded"] = le.fit_transform(train["Personality"])

In [60]:
train["Personality_encoded"].value_counts()

Personality_encoded
0    13699
1     4825
Name: count, dtype: int64

In [61]:
train["Personality_encoded"].value_counts()

Personality_encoded
0    13699
1     4825
Name: count, dtype: int64

### Take mode and median to fill NaN

In [62]:
numeric_columns = test.select_dtypes(include=['float64']).columns
for column in numeric_columns:
    train[column] = train[column].fillna(train[column].median())
    test[column] = test[column].fillna(test[column].median())

object_columns = test.select_dtypes(include=['object']).columns
for column in object_columns:
    train[column] = train[column].fillna(train[column].mode()[0])
    test[column] = test[column].fillna(test[column].mode()[0])

### Encode categorical features

In [63]:
ordinal_encoder = OrdinalEncoder()

train[object_columns] = ordinal_encoder.fit_transform(train[object_columns])
test[object_columns] = ordinal_encoder.transform(test[object_columns])

In [64]:
X = train.drop(columns=["id", "Personality", "Personality_encoded"])
y = train["Personality_encoded"]

In [65]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [66]:
X_train

,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency
1799,1.0,0.0,7.0,4.0,0.0,10.0,5.0
11931,2.0,0.0,4.0,6.0,0.0,6.0,8.0
14307,4.0,0.0,5.0,5.0,0.0,7.0,6.0
12157,3.0,0.0,6.0,4.0,0.0,8.0,8.0
18124,2.0,0.0,7.0,7.0,0.0,15.0,4.0
...,...,...,...,...,...,...,...
11284,9.0,0.0,1.0,3.0,1.0,5.0,3.0
11964,3.0,0.0,6.0,6.0,0.0,8.0,3.0
5390,3.0,0.0,7.0,3.0,0.0,14.0,8.0
860,3.0,0.0,4.0,4.0,0.0,9.0,9.0


## fit the classifier on training data ## 

In [90]:
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()

my_tree = MyDecisionTreeClassifier(max_depth = 5)
my_tree.fit(X_train_np, y_train_np)

In [91]:
X_test_np = X_test.to_numpy()
y_test_np = y_test.to_numpy()

predictions = my_tree.predict(X_test_np)

accuracy = accuracy_score(y_test_np, predictions)
print(f"Accuracy: {accuracy}")

Accuracy: 0.9678812415654521


In [92]:
np.bincount(predictions)

array([2764,  941])

In [75]:
tree_graph = my_tree.plot_tree()
tree_graph.view()

Error: Digraph.gv: syntax error in line 2 scanning a quoted string (missing endquote? longer than 16384?)
String starting:"leaf_TreeNode(value = None, left = TreeNode(value = None, left = TreeNode(value 


CalledProcessError: Command '[PosixPath('dot'), '-Kdot', '-Tpdf', '-O', 'Digraph.gv']' returned non-zero exit status 1. [stderr: b'Error: Digraph.gv: syntax error in line 2 scanning a quoted string (missing endquote? longer than 16384?)\nString starting:"leaf_TreeNode(value = None, left = TreeNode(value = None, left = TreeNode(value \n']